In [ ]:
!pip install sentence-transformers chromadb groq pandas -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 91.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not current

In [ ]:
import pandas as pd
import chromadb

from sentence_transformers import SentenceTransformer
from groq import Groq
import os
print("All libraries imported succesfully")

All libraries imported succesfully


In [ ]:
GROQ_API_KEY = "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"
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
groq_client = Groq(api_key=GROQ_API_KEY)

print("Groq API client initialized.")
print("Note: If you see an authentication error later, double-check your API key")

Groq API client initialized.
Note: If you see an authentication error later, double-check your API key


In [ ]:
df = pd.read_csv('college_notes.csv')
print("Shape of dataset", df.shape)
print("\nColumn names", df.columns.tolist())
print("\nFirst 3 rows:")
print(df.head(3))

Shape of dataset (15, 4)

Column names ['note_id', 'subject', 'topic', 'content']

First 3 rows:
  note_id           subject          topic  \
0    N001  Data Engineering  ETL Pipelines   
1    N002  Data Engineering  SQL Databases   
2    N003  Data Engineering  Data Cleaning   

                                             content  
0  ETL stands for Extract Transform Load. It is t...  
1  A database is an organized collection of data ...  
2  Data cleaning involves fixing or removing inco...  


In [ ]:
print("Subjects in the dataset:")
print(df['subject'].value_counts())
print("\nSample of topics:")
print(df[['note_id', 'subject', 'topic']].to_string(index=False))
print("\nLength of content (no. of char) for each note:")
df['content_length'] = df['content'].apply(len)
print(df[['topic', 'content_length']].to_string(index=False))

Subjects in the dataset:
subject
Data Engineering      5
Machine Learning      5
Generative AI         3
Python Programming    2
Name: count, dtype: int64

Sample of topics:
note_id            subject                          topic
   N001   Data Engineering                  ETL Pipelines
   N002   Data Engineering                  SQL Databases
   N003   Data Engineering                  Data Cleaning
   N004   Data Engineering       APIs and Data Collection
   N005   Data Engineering           Big Data and PySpark
   N006   Machine Learning            Supervised Learning
   N007   Machine Learning               Model Evaluation
   N008   Machine Learning            Feature Engineering
   N009   Machine Learning                 Decision Trees
   N010   Machine Learning                  Random Forest
   N011      Generative AI          Large Language Models
   N012      Generative AI             Prompt Engineering
   N013      Generative AI Retrieval Augmented Generation
   N014 Python

In [ ]:
documents = df['content'].tolist()
ids = [f"note_{row['note_id']} " for row in df.to_dict('records')]
metadatas = [
    {"subject": row['subject'], "topic": row['topic']}
    for row in df.to_dict('records')
]
print(f"Total chunks prepared : {len(documents)}")
print(f"First document Id : {ids[0]}")
print(f"First metadata : {metadatas[0]}")
print(f"First 100 chars of doc:{documents[0][:100]}...")

Total chunks prepared : 15
First document Id : note_N001 
First metadata : {'subject': 'Data Engineering', 'topic': 'ETL Pipelines'}
First 100 chars of doc:ETL stands for Extract Transform Load. It is the process of collecting raw data from different sourc...


In [ ]:
print("Loading embedding model...")
print("(This may take 30-60 seconds on first run - model is being downloaded)")
print("(Subsequent runs will be faster as the model is cached)")

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("\nEmbedding model loaded successfully.")
test_embedding = embedding_model.encode("This is a test sentence.")

print(f"Test embedding shape: {test_embedding.shape}")
print(f"First 5 values of test embedding: {test_embedding[:5]}")

Loading embedding model...
(This may take 30-60 seconds on first run - model is being downloaded)
(Subsequent runs will be faster as the model is cached)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Embedding model loaded successfully.
Test embedding shape: (384,)
First 5 values of test embedding: [0.08429647 0.05795366 0.00449333 0.1058211  0.00708344]


In [ ]:
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name = "college_notes_rag")
print("ChromaDB client created")
print(f"Collection name:College_notes_rag")
print(f"Documetns in collection so far: {collection.count()}")

ChromaDB client created
Collection name:College_notes_rag
Documetns in collection so far: 0


In [ ]:
print("Generating embedding for all the 15 notes")
print("This may take 15-30 sec")
embeddings = embedding_model.encode(documents, show_progress_bar= True)
print(f"\n Embedding matrix shape: {embeddings.shape}")
embeddings_list = embeddings.tolist()
collection.add(
    documents = documents,
    embeddings = embeddings_list,
    ids = ids,
    metadatas = metadatas
)
print(f"\nDocuments successfully added to chromadb.")
print(f"\nTotal documents in collection:{collection.count()}")

Generating embedding for all the 15 notes
This may take 15-30 sec


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


 Embedding matrix shape: (15, 384)

Documents successfully added to chromadb.

Total documents in collection:15


In [ ]:
def retrieve_relevant_chunks(question, top_k = 3):
  """
  Returns:
    A dictinoary containing retrieved documents, distances, and metadata
  """
  question_embedding = embedding_model.encode(question).tolist()
  results = collection.query(
      query_embeddings = [question_embedding],
      n_results = top_k
  )
  return results
print("Retrieval function defined successfully")
print("Function: retrieve_relevant_chunks(question, top_k = 3)")

Retrieval function defined successfully
Function: retrieve_relevant_chunks(question, top_k = 3)


In [ ]:
test_question = "What is ETL and how does it work in data engineering?"
print(f"Test Question:{test_question}")
print("="*60)
results = retrieve_relevant_chunks(test_question, top_k =3)
print("\n Top 3 retrieved chunks:")
print("="*60)
for i, (doc, dist, meta) in enumerate(zip(
    results['documents'][0],
    results['distances'][0],
    results['metadatas'][0]
)):
  print(f"\nResult{i+1}:")
  print(f"Subject: {meta['subject']}")
  print(f"Topic: {meta['topic']}")
  print(f"Distance: {dist}")
  print(f"Content: {doc[:100]}...")

Test Question:What is ETL and how does it work in data engineering?

 Top 3 retrieved chunks:

Result1:
Subject: Data Engineering
Topic: ETL Pipelines
Distance: 0.22689151763916016
Content: ETL stands for Extract Transform Load. It is the process of collecting raw data from different sourc...

Result2:
Subject: Data Engineering
Topic: APIs and Data Collection
Distance: 1.0689690113067627
Content: An API or Application Programming Interface allows two software applications to talk to each other. ...

Result3:
Subject: Python Programming
Topic: Data Visualization
Distance: 1.337485432624817
Content: Data visualization is the process of representing data as charts graphs and visual formats. Python l...


In [ ]:
def generate_rag_answer(question, context):
  """
  Send the retrieved contex and question to the Groq LLM for answer generation.
  Parameters:
    question(str) : The user's question
    context(str) : The retrieved context chunks (formatted string)

  Returns:
    answer(str): The LLM's generated answer
  """
  system_prompt = """ You are a helpful academic assistant for engineering students.
  You will be ginven context retrived from a  college knowledge base, and a student's question.

RULES:
1. Answer ONLY using the information provided in the context below.
2. If the answer is not found in the context, say exactly:
    "I don;t have enough information in my knowledge base to answer this question"
3. Do not use your general training knowledge.
4. Keep answers clear, accurate, and beginner friendly.
5. Mention which source the information came from when possible.
  """
  user_prompt = f"""Context from Knowledge Base:
{context}
---
Student's Question:{question}
Please answer the question based only on the context provided above.
"""
  response = groq_client.chat.completions.create(
      model = "llama-3.1-8b-instant",
      messages = [
          {"role": "system", "content": system_prompt},
          {"role": "user", "content": user_prompt}
      ],
      temperature = 0.1,
      max_tokens = 500
  )

  answer = response.choices[0].message.content

  return answer

print("RAG generation function defined.")

RAG generation function defined.
